554 HW10 McClure

Part 1 - Creating Streaming Data Using Rate

Setup a data stream using the "rate" format.

Prior to starting the stream, set up a sequence of actions using appropriate functions from pyspark.sql.functions
that uses the rate data to:

* find the square root of the rate ‘value’
* find mod 4 of the rate ‘value’

To output this, create a writeStream that writes to ‘memory’ (format("memory")). Give the query a name
(queryName("...")) and start it!

Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the query name (spark.sql("select * from you_table_name").show()).

In [5]:
# Load Library
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

# Start Spark Session
spark = SparkSession.builder.appName("HW10").getOrCreate()

In [6]:
# 1. Setup a data stream using the "rate" format
rate_stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

# 2. Apply transformations: square root and mod 4
transformed_df = rate_stream_df.withColumn("sqrt_value", F.sqrt("value")) \
                               .withColumn("mod4_value", F.col("value") % 4)

# 3. Create a writeStream to memory
query_name = "rate_memory_table"
query = transformed_df.writeStream \
    .format("memory") \
    .queryName(query_name) \
    .outputMode("append") \
    .start()

print(f"Streaming query '{query_name}' started. Running for 30 seconds...")

# 4. Let it run for about 30 seconds
query.awaitTermination(30) # Wait for 30 seconds

# 5. Stop the query
if query.isActive:
    query.stop()
    print("Streaming query stopped.")

# 6. Output the entire table stored in the query name
print(f"Displaying results from table '{query_name}':")
spark.sql(f"SELECT * FROM {query_name}").show()

26/04/22 02:39:16 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ddab05da-4c3c-4dc6-bcbe-2262337f4740. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/22 02:39:16 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming query 'rate_memory_table' started. Running for 30 seconds...
Streaming query stopped.
Displaying results from table 'rate_memory_table':
+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-22 02:39:...|    0|               0.0|         0|
|2026-04-22 02:39:...|    1|               1.0|         1|
|2026-04-22 02:39:...|    2|1.4142135623730951|         2|
|2026-04-22 02:39:...|    3|1.7320508075688772|         3|
|2026-04-22 02:39:...|    4|               2.0|         0|
|2026-04-22 02:39:...|    5|  2.23606797749979|         1|
|2026-04-22 02:39:...|    6| 2.449489742783178|         2|
|2026-04-22 02:39:...|    7|2.6457513110645907|         3|
|2026-04-22 02:39:...|    8|2.8284271247461903|         0|
|2026-04-22 02:39:...|    9|               3.0|         1|
|2026-04-22 02:39:...|   10|3.1622776601683795|         2|
|2026-04-22 02:39:...|   11

26/04/22 02:39:46 WARN DAGScheduler: Failed to cancel job group a2c24d9c-c3d8-46d7-b048-fcaee6fe8231. Cannot find active jobs for it.
26/04/22 02:39:46 WARN DAGScheduler: Failed to cancel job group a2c24d9c-c3d8-46d7-b048-fcaee6fe8231. Cannot find active jobs for it.


Part 2 - Using data from a CSV with a Pipeline

There are six bikeDetails sub datasets available on the assignment link. The one named bikeDetails_for_fit.csv
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following

* use an SQLTransformer with the following statement (this does some log transforms, renames a variable, and creates a dummy variable from categorical variable):

SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven, CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner FROM __THIS__

* use a VectorAssembler to create a features column. The features column should include the year,
log_km_driven, and one_owner variables.

* create a Pipeline with the two steps above (SQLTransformer then VectorAssembler)

* fit this pipeline to the SQL data frame and save this as an object.

Now we want to set up a read stream to look for csv files placed into a folder (the five bikeDetails_add*.csv
files). When a csv comes in, we want to transform it using the fitted pipeline’s .transform() method! A
few notes:

* You’ll need a schema to set up the readStream. You can use the SQL data frame’s schema from above!
(.schema attribute)

* Each file you’ll be adding to the folder has a header!

We’ll just write the output to the ‘console’ using the ‘append’ mode.

Once that is all set up, make sure your folder you are looking for the .csv files is empty. Then start the
query. Place the other files into the folder one at a time. You should see the output appear below your
query. Once you’ve done all five files, stop the query!

This process will set us up well for using a fitted model to obtain predictions on streaming data!

In [7]:
# 1. Read bikeDetails_for_fit.csv into a Spark DataFrame
csv_path = 'bike_database/bikeDetails_for_fit.csv'

# Infer schema to get the correct types for the initial DataFrame
initial_df = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)

print("Initial DataFrame schema:")
initial_df.printSchema()
initial_df.show(5)

# 2. Use an SQLTransformer
sql_statement = "SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven, CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner FROM __THIS__"
sql_transformer = SQLTransformer(statement=sql_statement)

# 3. Use a VectorAssembler
vector_assembler = VectorAssembler(inputCols=["year", "log_km_driven", "one_owner"], outputCol="features")

# 4. Create a Pipeline
pipeline = Pipeline(stages=[sql_transformer, vector_assembler])

# 5. Fit this pipeline to the SQL data frame
fitted_pipeline = pipeline.fit(initial_df)

print("\nPipeline fitted successfully.")

# Apply the fitted pipeline to the initial_df to show an example of transformation
transformed_initial_df = fitted_pipeline.transform(initial_df)
print("\nTransformed Initial DataFrame (first 5 rows):")
transformed_initial_df.select("label", "year", "log_km_driven", "one_owner", "features").show(5)

Initial DataFrame schema:
root
 |-- name: string (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- seller_type: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- km_driven: integer (nullable = true)
 |-- ex_showroom_price: integer (nullable = true)

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        200

In [ ]:
# 6. You'll need a schema to set up the readStream. Use the SQL data frame’s schema from above!
bike_schema = StructType([
    StructField("name", StringType(), True),
    StructField("selling_price", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("seller_type", StringType(), True),
    StructField("owner", StringType(), True),
    StructField("km_driven", IntegerType(), True),
    StructField("ex_showroom_price", IntegerType(), True)
])

streaming_schema = initial_df.schema
print("\nStreaming Schema (derived from initial_df):")
print(streaming_schema)

# Create a directory to monitor for new CSV files (if it doesn't exist)
import os
if not os.path.exists("bike_database"): # Using a dedicated folder for clarity
    os.makedirs("bike_database")
    print("Created directory: bike_database")

# 7. Set up a read stream to look for csv files placed into a folder
streaming_df = (spark.readStream
                .format("csv")
                .option("header", "true")
                .schema(bike_schema)
                .load("bike_database/")) # Monitor this folder

# 8. When a csv comes in, transform it using the fitted pipeline’s .transform() method!
streaming_transformed_df = fitted_pipeline.transform(streaming_df)

# 9. Write the output to the ‘console’ using the ‘append’ mode.
query_console = streaming_transformed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

query_console.awaitTermination() #can be used to block until the query is stopped manually

# End Goal is for all BikeDetails_add*.csv and BikeDetails_for_Fit.csv to exist in the same folder


Streaming Schema (derived from initial_df):
StructType([StructField('name', StringType(), True), StructField('selling_price', IntegerType(), True), StructField('year', IntegerType(), True), StructField('seller_type', StringType(), True), StructField('owner', StringType(), True), StructField('km_driven', IntegerType(), True), StructField('ex_showroom_price', IntegerType(), True)])


26/04/22 02:53:07 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d5f9afa8-a55d-4022-a478-cbc89e5d8f4b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/22 02:53:07 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
|12.072541252905651|2019| 5.857933154483459|        1|[2019.0,5.8579331...|
|10.714417768752456|2017| 8.639410824140487|        1|[2017.0,8.6394108...|
|11.918390573078392|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|11.082142548877775|2015|10.043249494911286|        1|[2015.0,10.043249...|
| 9.903487552536127|2011|  9.95227771670556|        0|[2011.0,9.9522777...|
| 9.798127036878302|2010|11.002099841204238|        1|[2010.0,11.002099...|
|  11.2708539037705|2018| 9.740968623038354|        1|[2018.0,9.7409686...|
|12.100712129872347|2008|10.571316925111784|        0|[2008.0,10.571316...|
|10.308952660644293|2010|10.373491181781864|        1|[2010.0,10.37